In [1]:


# our imports
# python imports
from crim_intervals import * 
from crim_intervals import importScore 
from crim_intervals import main_objs
from ipywidgets import interact
from pandas.io.json import json_normalize
from pyvis.network import Network
import altair as alt
import glob as glob
import crim_intervals.visualizations as viz
import numpy as np
import os
import pandas as pd
import re
import requests
from IPython.display import SVG
#from math import nan, isnan  # I think I don't need it anymore
from itertools import combinations # added this one too.

MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)

else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)

else:
    print(MUSDIR, "folder already exists.")
    

saved_csv folder already exists.
Music_Files folder already exists.


In [17]:
piece_list = ['CRIM_Model_0019.mei',
                     'CRIM_Mass_0019_1.mei',
                     'CRIM_Mass_0019_2.mei',
                     'CRIM_Mass_0019_3.mei',
                     'CRIM_Mass_0019_4.mei',
                     'CRIM_Mass_0019_5.mei']

In [3]:
for piece in piece_list:
    prefix = 'https://crimproject.org/mei/' 
# prefix = 'Music_Files/'
    mei_file = piece
    url = prefix + mei_file
    piece = importScore(url)
    

In [13]:
# settings for ngrams, unisons and kind

n=4
combineUnisons=False
kind='d'

In [5]:
from altair_saver import save

## an initial view of the Model, with all entries

In [18]:


# select the model from the list
model = piece_list[1]
prefix = 'https://crimproject.org/mei/' 
# prefix = 'Music_Files/'
url = prefix + model
model = importScore(url)

# find entries for model
nr = model.notes(combineUnisons=combineUnisons)
mel = model.melodic(df=nr, kind=kind, compound=False, unit=0, end=False)
# mel = model.melodic(df=nr, kind=kind, compound=False, end=False)

mod_mel_ngrams = model.ngrams(df=mel, n=n)
mod_entry_ngrams = model.entries(df=mel, n=n, thematic=True, anywhere=True)
mod_mel_ngrams_duration = model.durations(df=mel, n=n, mask_df=mod_entry_ngrams)
mod_entries_stack = list(mod_entry_ngrams.stack().unique())

print(model.metadata)

display(viz.plot_ngrams_heatmap(mod_entry_ngrams, mod_mel_ngrams_duration, 
                        selected_patterns=mod_entries_stack,
                        voices=[]))
mod_entry_ngrams

{'title': 'Missa Veni sponsa Christi: Kyrie', 'composer': 'Giovanni Pierluigi da Palestrina', 'date': 1599}


alt.Chart(...)

,Cantus,Altus,Tenor,Bassus
0.0,"(-3, 3, 2, -2)",NaN,NaN,NaN
8.0,NaN,"(-3, 3, 2, -2)",NaN,NaN
16.0,"(-2, -3, 2, 2)",NaN,NaN,NaN
24.0,"(2, 2, -2, -3)","(-2, -3, 2, 2)",NaN,NaN
26.0,"(-2, -3, 2, 2)",NaN,NaN,NaN
...,...,...,...,...
500.0,NaN,NaN,NaN,"(-2, 2, 1, 3)"
502.0,NaN,"(-2, 2, 2, 2)",NaN,NaN
510.0,NaN,NaN,"(-2, 2, 2, 2)",NaN
514.0,"(-2, 2, 1, 3)",NaN,NaN,NaN


## Now the Model Paired with Each Mass Movement

Showing only the 'shared' entries in each pair

In [19]:
# select the model from the list
model = piece_list[0]
prefix = 'https://crimproject.org/mei/' 
# prefix = 'Music_Files/'
url = prefix + model
model = importScore(url)

# find entries for model
nr = model.notes(combineUnisons=combineUnisons)
mel = model.melodic(df=nr, kind=kind, compound=False, unit=0, end=False)
mod_mel_ngrams = model.ngrams(df=mel, n=n)
mod_entry_ngrams = model.entries(df=mel, n=n, thematic=True, anywhere=True)
mod_mel_ngrams_duration = model.durations(df=mel, n=n, mask_df=mod_entry_ngrams)
mod_entries_stack = list(mod_entry_ngrams.stack().unique())

# find entries mass movements:
mass_movements = piece_list[1:6]

for movement in mass_movements:
    
    prefix = 'https://crimproject.org/mei/' 
    url = prefix + movement
    m_movement = importScore(url)
    nr = m_movement.notes(combineUnisons=combineUnisons)
    mel = m_movement.melodic(df=nr, kind=kind, compound=False, unit=0, end=False)
    mass_mvmt_mel_ngrams = m_movement.ngrams(df=mel, n=n)
    mass_mvmt_entries = m_movement.entries(df=mel, n=n, thematic=True, anywhere=True)
    mass_mvmt_mel_ngrams_duration = m_movement.durations(df=mel, n=n, mask_df=mass_mvmt_entries)
    mass_mvmt_entries_stack = mass_mvmt_entries.stack()

    
    # find shared entries
    
    shared_entries = list(set(mass_mvmt_entries_stack).intersection(mod_entries_stack))

    # print model metadata and chart
    print(model.metadata)

    display(viz.plot_ngrams_heatmap(mod_entry_ngrams, mod_mel_ngrams_duration, 
                        selected_patterns=shared_entries,
                        voices=[])) #.plot_ngrams_heatmap(entry_ngrams,


    # print mass movement metadata and chart
    print(m_movement.metadata)
    display(viz.plot_ngrams_heatmap(mass_mvmt_entries, mass_mvmt_mel_ngrams_duration, 
                                selected_patterns=shared_entries,
                                voices=[])) #.plot_ngrams_heatmap(entry_ngrams,


{'title': 'Veni sponsa Christi', 'composer': 'Giovanni Pierluigi da Palestrina', 'date': 1570}


alt.Chart(...)

{'title': 'Missa Veni sponsa Christi: Kyrie', 'composer': 'Giovanni Pierluigi da Palestrina', 'date': 1599}


alt.Chart(...)

{'title': 'Veni sponsa Christi', 'composer': 'Giovanni Pierluigi da Palestrina', 'date': 1570}


...

alt.Chart(...)

{'title': 'Veni sponsa Christi', 'composer': 'Giovanni Pierluigi da Palestrina', 'date': 1570}


alt.Chart(...)

{'title': 'Missa Veni sponsa Christi: Agnus Dei', 'composer': 'Giovanni Pierluigi da Palestrina', 'date': 1599}


alt.Chart(...)